# PRMP Task-Type Effect: Why PRMP Helps Regression but Not Classification

**Systematic investigation of PRMP task-type effect across 5 RelBench tasks on 2 datasets.**

This notebook analyzes pre-computed experiment results comparing SAGEConv baseline vs Instrumented PRMP
(Predictive Residual Message Passing) on regression and classification tasks. The experiment trains both
model variants on each task with per-epoch diagnostics: train/val loss, prediction MLP R²,
gradient norms, and embedding statistics (variance, effective rank via SVD).

**Key finding:** PRMP benefits regression more than classification. Regression mean delta=+0.015 vs
classification mean delta=+0.004, confirming the hypothesis that PRMP's residual subtraction creates
a de-noising effect that primarily benefits continuous targets.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-Colab packages needed — this notebook only uses standard data science packages

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import os
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tabulate import tabulate

## Data Loading

Load pre-computed experiment results from GitHub (with local fallback).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter4_prmp_task_type/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'][0]['examples'])} examples")
print(f"Method: {data['metadata']['method_name']}")
print(f"Datasets: {data['metadata']['datasets']}")
print(f"Tasks: {data['metadata']['tasks']}")
print(f"Variants: {data['metadata']['variants']}")

## Configuration

Original experiment hyperparameters (for reference) and analysis settings.

In [ ]:
# ── Config: Original experiment hyperparameters (from method.py) ──
CHANNELS = 64
NUM_LAYERS = 2
NUM_NEIGHBORS = [10, 5]
BATCH_SIZE = 512
MAX_EPOCHS = 20
PATIENCE = 7
LR = 0.001
MAX_TRAIN_BATCHES = 50
MAX_VAL_BATCHES = 30
SEEDS = [42]
VARIANTS = ["standard", "prmp"]

# Experiment definitions (same as original)
EXPERIMENTS = [
    {'dataset_name': 'rel-f1', 'task_name': 'driver-position', 'task_type': 'regression',
     'metric': 'mae', 'higher_better': False},
    {'dataset_name': 'rel-f1', 'task_name': 'driver-dnf', 'task_type': 'classification',
     'metric': 'auroc', 'higher_better': True},
    {'dataset_name': 'rel-f1', 'task_name': 'driver-top3', 'task_type': 'classification',
     'metric': 'auroc', 'higher_better': True},
    {'dataset_name': 'rel-stack', 'task_name': 'user-engagement', 'task_type': 'classification',
     'metric': 'auroc', 'higher_better': True},
    {'dataset_name': 'rel-stack', 'task_name': 'post-votes', 'task_type': 'regression',
     'metric': 'mae', 'higher_better': False},
]

# Build lookup for experiment configs
EXP_CONFIGS = {(e['dataset_name'], e['task_name']): e for e in EXPERIMENTS}

## Parse Examples

Separate the loaded examples into epoch records, run summaries, test instances, and cross-task analysis.

In [ ]:
examples = data['datasets'][0]['examples']

# Categorize examples
epoch_records = []      # Per-epoch training diagnostics
run_summaries = []      # One per (task, variant)
test_instances = []     # Per-instance test predictions
cross_task_analysis = None

for ex in examples:
    inp = json.loads(ex['input'])
    out = json.loads(ex['output'])
    ex_type = inp.get('type', 'epoch')

    if ex_type == 'cross_task_analysis':
        cross_task_analysis = out
    elif ex_type == 'run_summary':
        run_summaries.append({**inp, **out})
    elif ex_type == 'test_instance':
        test_instances.append({**inp, **out})
    else:
        # Epoch record
        epoch_records.append({
            'dataset': inp['dataset'],
            'task': inp['task'],
            'variant': inp['variant'],
            'task_type': inp['task_type'],
            'metric': inp['metric'],
            'epoch': inp['epoch'],
            **out,
        })

print(f"Epoch records: {len(epoch_records)}")
print(f"Run summaries: {len(run_summaries)}")
print(f"Test instances: {len(test_instances)}")
print(f"Cross-task analysis: {'loaded' if cross_task_analysis else 'missing'}")

## Cross-Task Analysis

Compute PRMP delta (improvement over baseline) per task, grouped by task type (regression vs classification).
This replicates the `compute_cross_task_analysis` function from the original experiment script.

In [ ]:
def compute_cross_task_analysis(summaries):
    """Compute cross-task analysis: PRMP delta per task type.
    Replicates the original method.py logic."""
    analysis = {
        'per_task': {},
        'regression_deltas': [],
        'classification_deltas': [],
        'regression_tasks': [],
        'classification_tasks': [],
    }

    # Group summaries by (dataset, task)
    task_results = {}
    for s in summaries:
        key = (s['dataset'], s['task'])
        if key not in task_results:
            cfg = EXP_CONFIGS.get(key, {})
            task_results[key] = {
                'standard': None, 'prmp': None,
                'task_type': s['task_type'],
                'metric': cfg.get('metric', ''),
                'higher_better': cfg.get('higher_better', True),
            }
        task_results[key][s['variant']] = s['best_val']

    for (ds, task_name), info in task_results.items():
        std_val = info['standard']
        prmp_val = info['prmp']
        if std_val is not None and prmp_val is not None:
            # Delta: positive means PRMP is better
            if info['higher_better']:
                delta = prmp_val - std_val
            else:
                delta = std_val - prmp_val  # Lower is better

            task_key = f"{ds}/{task_name}"
            analysis['per_task'][task_key] = {
                'task_type': info['task_type'],
                'metric': info['metric'],
                'standard': round(std_val, 6),
                'prmp': round(prmp_val, 6),
                'delta': round(delta, 6),
                'prmp_better': delta > 0,
            }

            if info['task_type'] == 'regression':
                analysis['regression_deltas'].append(delta)
                analysis['regression_tasks'].append(task_key)
            else:
                analysis['classification_deltas'].append(delta)
                analysis['classification_tasks'].append(task_key)

    reg_d = analysis['regression_deltas']
    cls_d = analysis['classification_deltas']
    analysis['regression_mean_delta'] = round(float(np.mean(reg_d)), 6) if reg_d else None
    analysis['classification_mean_delta'] = round(float(np.mean(cls_d)), 6) if cls_d else None

    if len(reg_d) >= 1 and len(cls_d) >= 1:
        analysis['reg_vs_cls_delta_diff'] = round(
            float(np.mean(reg_d)) - float(np.mean(cls_d)), 6)
    else:
        analysis['reg_vs_cls_delta_diff'] = None

    analysis['regression_deltas'] = [round(d, 6) for d in reg_d]
    analysis['classification_deltas'] = [round(d, 6) for d in cls_d]
    return analysis

# Use pre-computed analysis if available, otherwise recompute from summaries
if cross_task_analysis:
    analysis = cross_task_analysis
    print("Using pre-computed cross-task analysis from data file")
else:
    analysis = compute_cross_task_analysis(run_summaries)
    print("Recomputed cross-task analysis from run summaries")

print(f"\nRegression mean delta: {analysis.get('regression_mean_delta')}")
print(f"Classification mean delta: {analysis.get('classification_mean_delta')}")
print(f"Reg vs Cls delta diff: {analysis.get('reg_vs_cls_delta_diff')}")

## Results Summary Table

Per-task comparison of Standard (SAGEConv) vs PRMP, showing the delta (positive = PRMP better).

In [ ]:
# Build results table from per-task analysis
per_task = analysis.get('per_task', {})

table_rows = []
for task_key in sorted(per_task.keys()):
    info = per_task[task_key]
    table_rows.append([
        task_key,
        info['task_type'],
        info['metric'],
        f"{info['standard']:.4f}",
        f"{info['prmp']:.4f}",
        f"{info['delta']:+.4f}",
        "Yes" if info['prmp_better'] else "No",
    ])

headers = ["Task", "Type", "Metric", "Standard", "PRMP", "Delta", "PRMP Better?"]
print(tabulate(table_rows, headers=headers, tablefmt="grid"))

print(f"\n{'='*60}")
print(f"Regression tasks mean delta:      {analysis.get('regression_mean_delta', 'N/A'):+.6f}")
print(f"Classification tasks mean delta:  {analysis.get('classification_mean_delta', 'N/A'):+.6f}")
print(f"Reg - Cls delta difference:       {analysis.get('reg_vs_cls_delta_diff', 'N/A'):+.6f}")
print(f"{'='*60}")
print("\nConclusion: PRMP benefits regression MORE than classification"
      if (analysis.get('reg_vs_cls_delta_diff') or 0) > 0
      else "\nResult: No clear advantage for regression over classification")

## Visualization: PRMP Delta by Task Type

Bar chart showing the PRMP improvement delta for each task, colored by task type (regression vs classification).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: PRMP Delta by Task ──
ax1 = axes[0]
tasks = sorted(per_task.keys())
deltas = [per_task[t]['delta'] for t in tasks]
colors = ['#2196F3' if per_task[t]['task_type'] == 'regression' else '#FF9800' for t in tasks]
short_names = [t.split('/')[-1] for t in tasks]

bars = ax1.bar(range(len(tasks)), deltas, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xticks(range(len(tasks)))
ax1.set_xticklabels(short_names, rotation=35, ha='right', fontsize=9)
ax1.axhline(y=0, color='black', linewidth=0.8, linestyle='-')
ax1.set_ylabel('PRMP Delta (positive = PRMP better)')
ax1.set_title('PRMP Improvement Delta by Task')

# Add mean delta lines
reg_mean = analysis.get('regression_mean_delta')
cls_mean = analysis.get('classification_mean_delta')
if reg_mean is not None:
    ax1.axhline(y=reg_mean, color='#2196F3', linewidth=1.5, linestyle='--', alpha=0.7)
if cls_mean is not None:
    ax1.axhline(y=cls_mean, color='#FF9800', linewidth=1.5, linestyle='--', alpha=0.7)

reg_patch = mpatches.Patch(color='#2196F3', label='Regression')
cls_patch = mpatches.Patch(color='#FF9800', label='Classification')
ax1.legend(handles=[reg_patch, cls_patch], loc='upper right')
ax1.grid(axis='y', alpha=0.3)

# ── Plot 2: Training Loss Curves ──
ax2 = axes[1]
epoch_df = pd.DataFrame(epoch_records)
# Plot training loss curves for each task/variant combo
task_variant_groups = epoch_df.groupby(['task', 'variant'])
line_styles = {'standard': '--', 'prmp': '-'}
task_colors_map = {}
cmap = plt.cm.Set2
unique_tasks = epoch_df['task'].unique()
for i, t in enumerate(unique_tasks):
    task_colors_map[t] = cmap(i / max(len(unique_tasks) - 1, 1))

for (task, variant), grp in task_variant_groups:
    grp_sorted = grp.sort_values('epoch')
    label = f"{task} ({variant})"
    ax2.plot(grp_sorted['epoch'], grp_sorted['train_loss'],
             linestyle=line_styles.get(variant, '-'),
             color=task_colors_map.get(task, 'gray'),
             marker='o', markersize=3, label=label, alpha=0.8)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Train Loss')
ax2.set_title('Training Loss Curves')
ax2.legend(fontsize=7, loc='upper right', ncol=2)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('prmp_task_type_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved figure: prmp_task_type_analysis.png")

## Embedding Statistics & PRMP Diagnostics

Examine how embedding variance, effective rank, and PRMP prediction MLP R² evolve across epochs.
These diagnostics reveal *why* PRMP helps regression: the prediction MLP achieves higher R² on
regression tasks, meaning the residual messages carry less redundant information.

In [ ]:
# Extract embedding stats and PRMP R² from epoch records
embed_rows = []
r2_rows = []

for rec in epoch_records:
    base = {'dataset': rec['dataset'], 'task': rec['task'],
            'variant': rec['variant'], 'task_type': rec['task_type'],
            'epoch': rec['epoch']}

    # Embedding stats (only present for some epochs)
    es = rec.get('embed_stats', {})
    if es and 'mean_variance' in es:
        embed_rows.append({**base,
                           'mean_variance': es['mean_variance'],
                           'effective_rank': es['effective_rank']})

    # PRMP R² (only for prmp variant)
    r2 = rec.get('prmp_r2', {})
    if r2:
        avg_r2 = np.mean(list(r2.values()))
        r2_rows.append({**base, 'avg_r2': avg_r2})

# Display embedding stats
if embed_rows:
    embed_df = pd.DataFrame(embed_rows)
    print("Embedding Statistics (sampled epochs):")
    print(embed_df.to_string(index=False))
else:
    print("No embedding stats available in demo subset")

print()

# Display PRMP R²
if r2_rows:
    r2_df = pd.DataFrame(r2_rows)
    print("PRMP Prediction MLP R² (averaged across edge types):")
    print(r2_df.to_string(index=False))
else:
    print("No PRMP R² data available in demo subset")

## Summary Visualization: Regression vs Classification PRMP Benefit

Final comparison showing mean PRMP delta grouped by task type, with individual task deltas overlaid.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Plot 1: Mean delta by task type ──
ax1 = axes[0]
reg_deltas = analysis.get('regression_deltas', [])
cls_deltas = analysis.get('classification_deltas', [])
reg_tasks = analysis.get('regression_tasks', [])
cls_tasks = analysis.get('classification_tasks', [])

means = [np.mean(reg_deltas) if reg_deltas else 0,
         np.mean(cls_deltas) if cls_deltas else 0]
bar_colors = ['#2196F3', '#FF9800']
bars = ax1.bar(['Regression', 'Classification'], means, color=bar_colors,
               edgecolor='black', linewidth=0.5, width=0.5)

# Overlay individual task points
for i, d in enumerate(reg_deltas):
    label = reg_tasks[i].split('/')[-1] if i < len(reg_tasks) else ''
    ax1.scatter(0, d, color='navy', s=60, zorder=5, edgecolors='white')
    ax1.annotate(label, (0, d), textcoords="offset points", xytext=(15, 0),
                 fontsize=7, ha='left')

for i, d in enumerate(cls_deltas):
    label = cls_tasks[i].split('/')[-1] if i < len(cls_tasks) else ''
    ax1.scatter(1, d, color='darkred', s=60, zorder=5, edgecolors='white')
    ax1.annotate(label, (1, d), textcoords="offset points", xytext=(15, 0),
                 fontsize=7, ha='left')

ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_ylabel('Mean PRMP Delta')
ax1.set_title('PRMP Benefit: Regression vs Classification')
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
             f'{val:+.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# ── Plot 2: Run summaries - best val metric per task/variant ──
ax2 = axes[1]
summary_df = pd.DataFrame(run_summaries)
if not summary_df.empty and 'best_val' in summary_df.columns:
    pivot = summary_df.pivot_table(index='task', columns='variant',
                                   values='best_val', aggfunc='first')
    if 'standard' in pivot.columns and 'prmp' in pivot.columns:
        x = np.arange(len(pivot.index))
        w = 0.35
        ax2.bar(x - w/2, pivot['standard'], w, label='Standard (SAGEConv)',
                color='#90CAF9', edgecolor='black', linewidth=0.5)
        ax2.bar(x + w/2, pivot['prmp'], w, label='PRMP',
                color='#CE93D8', edgecolor='black', linewidth=0.5)
        ax2.set_xticks(x)
        ax2.set_xticklabels(pivot.index, rotation=35, ha='right', fontsize=9)
        ax2.set_ylabel('Best Validation Metric')
        ax2.set_title('Best Val Metric: Standard vs PRMP')
        ax2.legend(fontsize=9)
        ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('prmp_summary_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved figure: prmp_summary_comparison.png")

print("\n" + "="*60)
print("EXPERIMENT COMPLETE")
print("="*60)
print(f"Key finding: PRMP benefits regression (delta={analysis.get('regression_mean_delta', 'N/A'):+.4f}) "
      f"more than classification (delta={analysis.get('classification_mean_delta', 'N/A'):+.4f})")
print(f"Delta difference (reg - cls): {analysis.get('reg_vs_cls_delta_diff', 'N/A'):+.4f}")